### Elsevier data

In [1]:
import pandas as pd
import numpy as np
import requests
import json

In [146]:
data = pd.read_csv("./extracted_data/solar_energy_materials_and_solar_cells_canada.csv", index_col = False)

In [35]:
def get_citations_from_doi(doi):
    url = f"https://api.crossref.org/works/{doi}"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        citations_count = data["message"]["is-referenced-by-count"]
        return citations_count
    else:
        return None

In [147]:
data['Citations (Crossref)'] = data['Item DOI'].apply(get_citations_from_doi)

In [142]:
from pybliometrics.scopus import AuthorRetrieval, AbstractRetrieval, PlumXMetrics
import pybliometrics.scopus



pybliometrics.scopus.utils.create_config(keys=[__import__("os").environ["SCOPUS_API_KEY"]])

Creating config file at C:\Users\aznur\.config\pybliometrics.cfg with default paths...
Configuration file successfully created at C:\Users\aznur\.config\pybliometrics.cfg
For details see https://pybliometrics.rtfd.io/en/stable/configuration.html.


In [143]:
pybliometrics.scopus.init()

def get_author_info(doi):
    try:
        ab = AbstractRetrieval(doi)
        first_author = ab.authors[0].auid
        coauthors = ab.authors[1:]

        first_author_info = AuthorRetrieval(first_author)
        first_author_name = f"{first_author_info.given_name} {first_author_info.surname}"
        first_author_h_index = first_author_info.h_index
        first_author_citation_count = first_author_info.citation_count

        coauthors_h_indices = []
        coauthors_names = []
        for coauthor in coauthors:
            coauthor_info = AuthorRetrieval(coauthor.auid)
            coauthor_name = f"{coauthor_info.given_name} {coauthor_info.surname}"
            coauthor_h_index = coauthor_info.h_index
            coauthors_names.append(coauthor_name)
            coauthors_h_indices.append(coauthor_h_index)

        captures = 0
        citations = 0
        mentions = 0
        social_media = 0
        usages = 0

        plum = PlumXMetrics(doi, id_type='doi')
        for category in plum.category_totals:
            if category.name == 'capture':
                captures = category.total
            elif category.name == 'citation':
                citations = category.total
            elif category.name == 'mention':
                mentions = category.total
            elif category.name == 'socialMedia':
                social_media = category.total
            elif category.name == 'usage':
                usages = category.total

        return first_author_name, first_author_h_index, coauthors_names, coauthors_h_indices, captures, citations, mentions, social_media, usages
    except Exception as e:
        print(f"Error: {e}")
        return None


def fetch_and_update_author_info(row):
    doi = row['Item DOI']
    author_info = get_author_info(doi)
    if author_info is not None:
        first_author_name, first_author_h_index, coauthors_names, coauthors_h_indices, captures, citations, mentions, social_media, usages = author_info
        data.at[row.name, 'First Author'] = first_author_name
        data.at[row.name, 'First Author H-Index'] = first_author_h_index
        data.at[row.name, 'Authors'] = coauthors_names
        data.at[row.name, 'Authors H-index'] = ', '.join(map(str, coauthors_h_indices))
        data.at[row.name, 'Captures'] = captures
        data.at[row.name, 'Citations'] = citations
        data.at[row.name, 'Mentions'] = mentions
        data.at[row.name, 'Social Media'] = social_media
        data.at[row.name, 'Usages'] = usages
    else:
        print(f"No author information found for DOI: {doi}")

In [148]:
data.apply(fetch_and_update_author_info, axis=1)

c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.h

Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2024.113399


0      None
1      None
2      None
3      None
4      None
       ... 
114    None
115    None
116    None
117    None
118    None
Length: 119, dtype: object

In [149]:
data.to_csv('./elsevier_data/solar_energy_materials_and_solar_cells_canada.csv', index=False)

In [150]:
data = pd.read_csv("./extracted_data/solar_energy_materials_and_solar_cells_france.csv", index_col = False)

In [151]:
data['Citations (Crossref)'] = data['Item DOI'].apply(get_citations_from_doi)
data.apply(fetch_and_update_author_info, axis=1)
data.to_csv('./elsevier_data/solar_energy_materials_and_solar_cells_france.csv', index=False)

c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.h

Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2025.113415
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/j.solmat.2021.111255
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2025.113401
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2024.113371


In [152]:
data = pd.read_csv("./extracted_data/solar_energy_materials_and_solar_cells_germany.csv", index_col = False)

In [153]:
data['Citations (Crossref)'] = data['Item DOI'].apply(get_citations_from_doi)
data.apply(fetch_and_update_author_info, axis=1)
data.to_csv('./elsevier_data/solar_energy_materials_and_solar_cells_germany.csv', index=False)

c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.h

Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2024.113252
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2025.113432
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/j.solmat.2022.112044
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2024.113361
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/j.solmat.2021.111252


In [154]:
data = pd.read_csv("./extracted_data/solar_energy_materials_and_solar_cells_singapore.csv", index_col = False)

In [155]:
data['Citations (Crossref)'] = data['Item DOI'].apply(get_citations_from_doi)
data.apply(fetch_and_update_author_info, axis=1)
data.to_csv('./elsevier_data/solar_energy_materials_and_solar_cells_singapore.csv', index=False)

c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.h

Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2025.113427


In [156]:
data = pd.read_csv("./extracted_data/solar_energy_materials_and_solar_cells_south africa.csv", index_col = False)

In [157]:
data['Citations (Crossref)'] = data['Item DOI'].apply(get_citations_from_doi)
data.apply(fetch_and_update_author_info, axis=1)
data.to_csv('./elsevier_data/solar_energy_materials_and_solar_cells_south africa.csv', index=False)

c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.h

In [158]:
data = pd.read_csv("./extracted_data/solar_energy_materials_and_solar_cells_switzerland.csv", index_col = False)

In [159]:
data['Citations (Crossref)'] = data['Item DOI'].apply(get_citations_from_doi)
data.apply(fetch_and_update_author_info, axis=1)
data.to_csv('./elsevier_data/solar_energy_materials_and_solar_cells_switzerland.csv', index=False)

c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.h

Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/j.solmat.2019.01.033
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/j.solmat.2019.01.031


In [160]:
data = pd.read_csv("./extracted_data/solar_energy_materials_and_solar_cells_uk.csv", index_col = False)

In [161]:
data['Citations (Crossref)'] = data['Item DOI'].apply(get_citations_from_doi)
data.apply(fetch_and_update_author_info, axis=1)
data.to_csv('./elsevier_data/solar_energy_materials_and_solar_cells_uk.csv', index=False)

c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.h

Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2024.113385
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2025.113426
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2025.113440
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2024.113385
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2025.113426
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2025.113440


In [162]:
data = pd.read_csv("./extracted_data/solar_energy_materials_and_solar_cells_us.csv", index_col = False)

In [163]:
data['Citations (Crossref)'] = data['Item DOI'].apply(get_citations_from_doi)
data.apply(fetch_and_update_author_info, axis=1)
data.to_csv('./elsevier_data/solar_energy_materials_and_solar_cells_us.csv', index=False)

c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.h

Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2024.112722
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2024.113365


In [129]:
data = pd.read_csv("./extracted_data/solar_energy_canada.csv", index_col = False)
data['Citations (Crossref)'] = data['Item DOI'].apply(get_citations_from_doi)
data.apply(fetch_and_update_author_info, axis=1)
data.to_csv('./elsevier_data/solar_energy_canada.csv', index=False)

c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.h

Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solener.2025.113289
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2024.113399
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/B978-0-323-90601-2.00018-0
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/B978-0-12-811479-7.00030-0


In [132]:
# data = pd.read_csv("./extracted_data/solar_energy_france.csv", index_col = False)
# data['Citations (Crossref)'] = data['Item DOI'].apply(get_citations_from_doi)
data.apply(fetch_and_update_author_info, axis=1)
data.to_csv('./elsevier_data/solar_energy_france.csv', index=False)

c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])


Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2025.113415
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solener.2025.113250
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2025.113401
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solener.2024.113187
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solener.2024.113227
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solener.2024.113215
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2024.113371
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/j.solmat.2021.111255
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solener.2020.05.100
Error: The resource specif

In [136]:
data = pd.read_csv("./extracted_data/solar_energy_germany.csv", index_col = False)
data['Citations (Crossref)'] = data['Item DOI'].apply(get_citations_from_doi)
data.apply(fetch_and_update_author_info, axis=1)
data.to_csv('./elsevier_data/solar_energy_germany.csv', index=False)

c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.h

Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2024.113252
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solener.2024.112911
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2025.113432
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2024.113361
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/j.solmat.2022.112044
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/j.solmat.2021.111252
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/B978-0-12-823959-9.01002-3
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/B978-0-12-820072-8.01002-7
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/B978-0-323-8986

In [137]:
data = pd.read_csv("./extracted_data/solar_energy_singapore.csv", index_col = False)
data['Citations (Crossref)'] = data['Item DOI'].apply(get_citations_from_doi)
data.apply(fetch_and_update_author_info, axis=1)
data.to_csv('./elsevier_data/solar_energy_singapore.csv', index=False)

c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.h

Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2025.113427
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/B978-0-443-23989-2.00006-9
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/B978-0-443-23989-2.00023-9
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/j.solener.2021.03.051


In [138]:
data = pd.read_csv("./extracted_data/solar_energy_south africa.csv", index_col = False)
data['Citations (Crossref)'] = data['Item DOI'].apply(get_citations_from_doi)
data.apply(fetch_and_update_author_info, axis=1)
data.to_csv('./elsevier_data/solar_energy_south africa.csv', index=False)

c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.h

Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solener.2025.113263
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.seja.2025.100090
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solener.2024.113197
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solener.2024.113223
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/B978-0-12-409540-3.09992-8
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/B978-0-12-811479-7.00030-0


In [139]:
data = pd.read_csv("./extracted_data/solar_energy_switzerland.csv", index_col = False)
data['Citations (Crossref)'] = data['Item DOI'].apply(get_citations_from_doi)
data.apply(fetch_and_update_author_info, axis=1)
data.to_csv('./elsevier_data/solar_energy_switzerland.csv', index=False)

c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.h

Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2024.112722
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.seja.2024.100081
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/B978-0-443-23989-2.00012-4
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2024.113365
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/B978-0-443-23989-2.00023-9


In [144]:
data = pd.read_csv("./extracted_data/solar_energy_uk.csv", index_col = False)
data['Citations (Crossref)'] = data['Item DOI'].apply(get_citations_from_doi)
data.apply(fetch_and_update_author_info, axis=1)
data.to_csv('./elsevier_data/solar_energy_uk.csv', index=False)

c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.h

Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2024.113385
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2025.113426
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solener.2025.113293
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solener.2025.113267
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solmat.2025.113440
Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solener.2024.113213
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/B978-0-12-409540-3.09992-8


In [145]:
data = pd.read_csv("./extracted_data/solar_energy_us.csv", index_col = False)
data['Citations (Crossref)'] = data['Item DOI'].apply(get_citations_from_doi)
data.apply(fetch_and_update_author_info, axis=1)
data.to_csv('./elsevier_data/solar_energy_us.csv', index=False)

c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\aznur\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.h

Error: 'NoneType' object is not iterable
No author information found for DOI: 10.1016/j.solener.2025.113271
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/j.solmat.2019.01.033
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/j.solmat.2019.01.031
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/B978-0-12-823959-9.01002-3
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/B978-0-323-90601-2.00018-0
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/B978-0-323-89866-9.00014-6
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/B978-0-12-821592-0.00025-X
Error: The resource specified cannot be found.
No author information found for DOI: 10.1016/B978-0-12-819610-6.09996-8
Error: The resource specified cannot be found.
No author information fo

### Gender classification

In [41]:
import pandas as pd

data = pd.read_csv("./elsevier_data/applied_catalysis_germany.csv", index_col = False)

In [42]:
data["First Author"].fillna("N/A", inplace = True)

In [43]:
count = data['First Author'].value_counts()
if 'N/A' in count.index:
    na_count = count['N/A']
    print(na_count)

1


In [44]:
data = data.drop('Book Series Title', axis=1)

In [45]:
len(data)

282

In [46]:
import requests

# url = "https://v2.namsor.com/NamSorAPIv2/api2/json/originBatch"
url = "https://v2.namsor.com/NamSorAPIv2/api2/json/genderFullGeoBatch"

headers = {
    "X-API-KEY": __import__("os").environ["NAMSOR_API_KEY"],
    "Accept": "application/json",
    "Content-Type": "application/json"
}

In [48]:
new_column_values = []
new_column_values1 = []

for index, row in data.iterrows():
    value = row['First Author']

    payload = {
        "personalNames": [
            {
                "name": f"{value}",
                "countryIso2": "DE"
            }
        ]
    }

    response = requests.request("POST", url, json=payload, headers=headers)
    print(response.text)

    response_json = response.json()
    likely_gender = response_json['personalNames'][0]['likelyGender']
    probability = response_json['personalNames'][0]['probabilityCalibrated']
    new_column_values.append(likely_gender)
    new_column_values1.append(probability)
    

data['Gender (First Author)'] = new_column_values
data['Gender likelihood (First Author)'] = new_column_values1

{"personalNames":[{"script":"LATIN","id":"9ca6597c-4d38-4886-b851-678c7c32367f","name":"Ge Li","likelyGender":"male","genderScale":-0.3532703984359207,"score":3.0895371106428793,"probabilityCalibrated":0.6766351992179603}]}
{"personalNames":[{"script":"LATIN","id":"11e01c87-2438-4f0f-af76-16f372b0542a","name":"Saioa Goicoechea","likelyGender":"female","genderScale":0.43741042116893514,"score":5.477669777959193,"probabilityCalibrated":0.7187052105844676}]}
{"personalNames":[{"script":"LATIN","id":"7208edc5-e7d3-45cd-8f69-500475190bc7","name":"Dennis Großmann","likelyGender":"male","genderScale":-0.8133286144607121,"score":8.431923100248625,"probabilityCalibrated":0.9066643072303561}]}
{"personalNames":[{"script":"LATIN","id":"5d176fd8-4154-4a47-9153-14f3531b20ba","name":"Sebastian Werner","likelyGender":"male","genderScale":-0.8355833541195747,"score":8.654878223883784,"probabilityCalibrated":0.9177916770597874}]}
{"personalNames":[{"script":"LATIN","id":"18221834-240a-413f-b67e-33abba6

In [49]:
data.to_csv("./processed_data/applied_catalysis_germany_gender.csv", index = False)

In [50]:
new_column_values = []
new_column_values1 = []

for index, row in data.iterrows():
    if isinstance(row['Authors'], list):
        names = row['Authors']
    else:
        names = row['Authors'].split(', ')

    names_gender = []
    names_probability = []
    
    for name in names:
        payload = {
            "personalNames": [
                {
                    "name": name.strip(),
                    "countryIso2": "DE"
                }
            ]
        }

        response = requests.request("POST", url, json=payload, headers=headers)
        print(response.text)
        response_json = response.json()

        likely_gender = response_json['personalNames'][0]['likelyGender']
        probability = response_json['personalNames'][0]['probabilityCalibrated']
        
        names_gender.append(likely_gender)
        names_probability.append(probability)

    new_column_values.append(names_gender)
    new_column_values1.append(names_probability)

data['Gender (Authors)'] = new_column_values
data['Gender likelihood (Authors)'] = new_column_values1

{"personalNames":[{"script":"LATIN","id":"eecc5551-fc2b-472d-9ba3-f647ab214b77","name":"['Chunxiao Jin'","likelyGender":"male","genderScale":-0.2798723519075508,"score":2.0662172809816894,"probabilityCalibrated":0.6399361759537754}]}
{"personalNames":[{"script":"LATIN","id":"b1659f44-ddf5-48d4-a7c6-253377ee2746","name":"'Baodong Wang'","likelyGender":"female","genderScale":0.2812077309773853,"score":2.0787012697245757,"probabilityCalibrated":0.6406038654886926}]}
{"personalNames":[{"script":"LATIN","id":"695430d9-b800-4539-a171-8ea1a9750f01","name":"'Ziran Ma'","likelyGender":"female","genderScale":0.17209763904156694,"score":1.6131787382022367,"probabilityCalibrated":0.5860488195207835}]}
{"personalNames":[{"script":"LATIN","id":"38bd1c44-1d6b-4a58-bd41-72772cb649da","name":"'Pan Gao'","likelyGender":"female","genderScale":0.08640009281272443,"score":0.4912242180790401,"probabilityCalibrated":0.5432000464063622}]}
{"personalNames":[{"script":"LATIN","id":"ce4633d6-36bb-4b15-b747-1431f

In [51]:
data.to_csv("./processed_data/applied_catalysis_germany_genders.csv", index = False)

In [57]:
import pandas as pd
import requests
import os

countries = ["Canada", "France", "Germany", "Singapore", "South Africa", "Switzerland", "UK", "US"]
country_iso_mapping = {
    "Canada": "CA",
    "France": "FR",
    "Germany": "DE",
    "Singapore": "SG",
    "South Africa": "ZA", 
    "Switzerland": "CH",
    "UK": "GB",
    "US": "US"
}

url = "https://v2.namsor.com/NamSorAPIv2/api2/json/genderFullGeoBatch"
headers = {
    "X-API-KEY": __import__("os").environ["NAMSOR_API_KEY"],
    "Accept": "application/json",
    "Content-Type": "application/json"
}

for country in countries:
    print(f"Processing {country}...")
    country_iso = country_iso_mapping[country]
    
    country_lower = country.lower()
    filename = country_lower.replace(" ", "_")
    
    try:
        input_path = f"./elsevier_data/hydrogen_energy_{filename}.csv"
        data = pd.read_csv(input_path, index_col=False)
        
        data["First Author"].fillna("N/A", inplace=True)
        
        count = data['First Author'].value_counts()
        if 'N/A' in count.index:
            na_count = count['N/A']
            print(f"N/A count for {country}: {na_count}")
        
        if 'Book Series Title' in data.columns:
            data = data.drop('Book Series Title', axis=1)
        
        print(f"Total records for {country}: {len(data)}")
        
        new_column_values = []
        new_column_values1 = []
        
        for index, row in data.iterrows():
            value = row['First Author']
            
            payload = {
                "personalNames": [
                    {
                        "name": f"{value}",
                        "countryIso2": country_iso
                    }
                ]
            }
            
            response = requests.request("POST", url, json=payload, headers=headers)
            print(f"Processing {country} - First author {index+1}/{len(data)}")
            
            response_json = response.json()
            likely_gender = response_json['personalNames'][0]['likelyGender']
            probability = response_json['personalNames'][0]['probabilityCalibrated']
            new_column_values.append(likely_gender)
            new_column_values1.append(probability)
        
        data['Gender (First Author)'] = new_column_values
        data['Gender likelihood (First Author)'] = new_column_values1
        
        data.to_csv(f"./processed_data/hydrogen_energy_{filename}_gender.csv", index=False)
        print(f"Saved first author gender data for {country}")
        
        new_column_values = []
        new_column_values1 = []
        
        for index, row in data.iterrows():
            if isinstance(row['Authors'], list):
                names = row['Authors']
            else:
                names = row['Authors'].split(', ')
            
            names_gender = []
            names_probability = []
            
            for name in names:
                payload = {
                    "personalNames": [
                        {
                            "name": name.strip(),
                            "countryIso2": country_iso
                        }
                    ]
                }
                
                response = requests.request("POST", url, json=payload, headers=headers)
                print(f"Processing {country} - All authors - record {index+1}/{len(data)}")
                response_json = response.json()
                
                likely_gender = response_json['personalNames'][0]['likelyGender']
                probability = response_json['personalNames'][0]['probabilityCalibrated']
                
                names_gender.append(likely_gender)
                names_probability.append(probability)
            
            new_column_values.append(names_gender)
            new_column_values1.append(names_probability)
        
        data['Gender (Authors)'] = new_column_values
        data['Gender likelihood (Authors)'] = new_column_values1
        
        data.to_csv(f"./processed_data/hydrogen_energy_{filename}_genders.csv", index=False)
        print(f"Completed processing for {country}")
        
    except Exception as e:
        print(f"Error processing {country}: {str(e)}")
        continue
    
print("All countries processed successfully.")

Processing Canada...
N/A count for Canada: 16
Total records for Canada: 1505
Processing Canada - First author 1/1505
Processing Canada - First author 2/1505
Processing Canada - First author 3/1505
Processing Canada - First author 4/1505
Processing Canada - First author 5/1505
Processing Canada - First author 6/1505
Processing Canada - First author 7/1505
Processing Canada - First author 8/1505
Processing Canada - First author 9/1505
Processing Canada - First author 10/1505
Processing Canada - First author 11/1505
Processing Canada - First author 12/1505
Processing Canada - First author 13/1505
Processing Canada - First author 14/1505
Processing Canada - First author 15/1505
Processing Canada - First author 16/1505
Processing Canada - First author 17/1505
Processing Canada - First author 18/1505
Processing Canada - First author 19/1505
Processing Canada - First author 20/1505
Processing Canada - First author 21/1505
Processing Canada - First author 22/1505
Processing Canada - First auth

In [56]:
import pandas as pd
import requests
import os

countries = ["Canada", "France", "Germany", "Singapore", "South Africa", "Switzerland", "UK", "US"]
country_iso_mapping = {
    "Canada": "CA",
    "France": "FR",
    "Germany": "DE",
    "Singapore": "SG",
    "South Africa": "ZA", 
    "Switzerland": "CH",
    "UK": "GB",
    "US": "US"
}

url = "https://v2.namsor.com/NamSorAPIv2/api2/json/genderFullGeoBatch"
headers = {
    "X-API-KEY": __import__("os").environ["NAMSOR_API_KEY"],
    "Accept": "application/json",
    "Content-Type": "application/json"
}

for country in countries:
    print(f"Processing {country}...")
    country_iso = country_iso_mapping[country]
    
    country_lower = country.lower()
    filename = country_lower.replace(" ", "_")
    
    try:
        input_path = f"./elsevier_data/solar_energy_{filename}.csv"
        data = pd.read_csv(input_path, index_col=False)
        
        data["First Author"].fillna("N/A", inplace=True)
        
        count = data['First Author'].value_counts()
        if 'N/A' in count.index:
            na_count = count['N/A']
            print(f"N/A count for {country}: {na_count}")
        
        if 'Book Series Title' in data.columns:
            data = data.drop('Book Series Title', axis=1)
        
        print(f"Total records for {country}: {len(data)}")
        
        new_column_values = []
        new_column_values1 = []
        
        for index, row in data.iterrows():
            value = row['First Author']
            
            payload = {
                "personalNames": [
                    {
                        "name": f"{value}",
                        "countryIso2": country_iso
                    }
                ]
            }
            
            response = requests.request("POST", url, json=payload, headers=headers)
            print(f"Processing {country} - First author {index+1}/{len(data)}")
            
            response_json = response.json()
            likely_gender = response_json['personalNames'][0]['likelyGender']
            probability = response_json['personalNames'][0]['probabilityCalibrated']
            new_column_values.append(likely_gender)
            new_column_values1.append(probability)
        
        data['Gender (First Author)'] = new_column_values
        data['Gender likelihood (First Author)'] = new_column_values1
        
        data.to_csv(f"./processed_data/solar_energy_{filename}_gender.csv", index=False)
        print(f"Saved first author gender data for {country}")
        
        new_column_values = []
        new_column_values1 = []
        
        for index, row in data.iterrows():
            if isinstance(row['Authors'], list):
                names = row['Authors']
            else:
                names = row['Authors'].split(', ')
            
            names_gender = []
            names_probability = []
            
            for name in names:
                payload = {
                    "personalNames": [
                        {
                            "name": name.strip(),
                            "countryIso2": country_iso
                        }
                    ]
                }
                
                response = requests.request("POST", url, json=payload, headers=headers)
                print(f"Processing {country} - All authors - record {index+1}/{len(data)}")
                response_json = response.json()
                
                likely_gender = response_json['personalNames'][0]['likelyGender']
                probability = response_json['personalNames'][0]['probabilityCalibrated']
                
                names_gender.append(likely_gender)
                names_probability.append(probability)
            
            new_column_values.append(names_gender)
            new_column_values1.append(names_probability)
        
        data['Gender (Authors)'] = new_column_values
        data['Gender likelihood (Authors)'] = new_column_values1
        
        data.to_csv(f"./processed_data/solar_energy_{filename}_genders.csv", index=False)
        print(f"Completed processing for {country}")
        
    except Exception as e:
        print(f"Error processing {country}: {str(e)}")
        continue
    
print("All countries processed successfully.")

Processing Canada...
N/A count for Canada: 4
Total records for Canada: 366
Processing Canada - First author 1/366
Processing Canada - First author 2/366
Processing Canada - First author 3/366
Processing Canada - First author 4/366
Processing Canada - First author 5/366
Processing Canada - First author 6/366
Processing Canada - First author 7/366
Processing Canada - First author 8/366
Processing Canada - First author 9/366
Processing Canada - First author 10/366
Processing Canada - First author 11/366
Processing Canada - First author 12/366
Processing Canada - First author 13/366
Processing Canada - First author 14/366
Processing Canada - First author 15/366
Processing Canada - First author 16/366
Processing Canada - First author 17/366
Processing Canada - First author 18/366
Processing Canada - First author 19/366
Processing Canada - First author 20/366
Processing Canada - First author 21/366
Processing Canada - First author 22/366
Processing Canada - First author 23/366
Processing Can

In [57]:
def get_aff_info(doi):
    try:
        ab = AbstractRetrieval(doi)
        first_author = ab.authors[0].auid
        coauthors = ab.authors[1:]

        first_author_info = AuthorRetrieval(first_author)
        first_author_country = first_author_info.affiliation_current[0].country
        first_author_aff_id = first_author_info.affiliation_current[0].id

        coauthors_info = []
        coauthors_affiliation_countries = []
        coauthors_affiliation_ids = []
        
        for coauthor in coauthors:
            coauthor_info = AuthorRetrieval(coauthor.auid)
            coauthors_affiliation_countries.append(coauthor_info.affiliation_current[0].country)
            coauthors_affiliation_ids.append(coauthor_info.affiliation_current[0].id)

        return first_author_country, first_author_aff_id, coauthors_affiliation_countries, coauthors_affiliation_ids
            
    except Exception as e:
        print(f"Error: {e}")
        return None

In [59]:
data["First Author Country"] = ""
data["First Author Affiliation ID"] = ""
data["Coauthors Affiliation Countries"] = ""
data["Coauthors Affiliation IDs"] = ""

for index, row in data.iterrows():
    doi = row["Item DOI"]
    result = get_aff_info(doi)
    if result:
        first_author_country, first_author_aff_id, coauthors_affiliation_countries, coauthors_affiliation_ids = result

        data.at[index, "First Author Country"] = first_author_country
        data.at[index, "First Author Affiliation ID"] = first_author_aff_id
        data.at[index, "Coauthors Affiliation Countries"] = coauthors_affiliation_countries
        data.at[index, "Coauthors Affiliation IDs"] = coauthors_affiliation_ids

In [60]:
data

,Item Title,Publication Title,Journal Volume,Journal Issue,Item DOI,Authors,Publication Year,URL,Content Type,Citations (Crossref),...,Usages,first_author_gender,first_author_gender_probability,authors_genders,authors_gender_probabilities,Keywords,First Author Country,First Author Affiliation ID,Coauthors Affiliation Countries,Coauthors Affiliation IDs
0,Substitutional effect of Ti-based AB2 hydrogen...,International Journal of Hydrogen Energy,48.0,35,10.1016/j.ijhydene.2022.12.083,"['D. M. Grant', 'Gavin S. Walker', 'Sanliang L...",2023,https://www.sciencedirect.com/science/article/...,Article,0,...,0,male,0.628521,"['male', 'male', 'female']","[0.52807997057241, 0.9372461990030759, 0.66099...","Hydrogen storage, AB2 alloy, Substitution, For...",United Kingdom,103778130,"[United Kingdom, United Kingdom, United Kingdom]","[60015138, 60015138, 103778130]"
1,An iron ore-based catalyst for producing hydro...,International Journal of Hydrogen Energy,48.0,57,10.1016/j.ijhydene.2023.03.022,"['David S. Saal', 'J. F. Marco', 'James C. Rey...",2023,https://www.sciencedirect.com/science/article/...,Article,0,...,0,female,0.873772,"['male', 'male', 'male', 'female']","[0.9569404582479849, 0.5356845865798004, 0.961...","Catalytic methane pyrolysis, Iron ore, Hydroge...",United Kingdom,60000891,"[United Kingdom, Spain, United Kingdom, United...","[60116333, 60007643, 60000891, 60000891]"
2,Al2O3–H2O nanofluids for cooling PEM fuel cell...,International Journal of Hydrogen Energy,47.0,91,10.1016/j.ijhydene.2022.09.040,"['Maher A.R.Sadiq Al-Baghdadi', 'Ward F. Arear...",2022,https://www.sciencedirect.com/science/article/...,Article,3,...,0,male,0.756790,"['male', 'male', 'male']","[0.9626864795993925, 0.9343137048244043, 0.529...","Fuel cell, Cooling management, Hydrothermal pe...",Iraq,60071160,"[Iraq, Iraq, United Kingdom]","[60071160, 114593797, 60030469]"
3,Numerical simulation of hydrogen leakage diffu...,International Journal of Hydrogen Energy,48.0,63,10.1016/j.ijhydene.2023.03.208,"['Yupeng Yuan', 'Liang Tong', 'Boyang Shen']",2023,https://www.sciencedirect.com/science/article/...,Article,1,...,0,female,0.636018,"['male', 'male', 'male']","[0.670735628997693, 0.5352080988797604, 0.6787...","Hydrogen refueling station, Hydrogen leakage, ...",China,129436590,"[China, China, United Kingdom]","[104144923, 104144923, 60031101]"
4,A fast two-phase non-isothermal reduced-order ...,International Journal of Hydrogen Energy,47.0,91,10.1016/j.ijhydene.2022.09.044,"['Huizhi Wang', 'Nigel P. Brandon']",2022,https://www.sciencedirect.com/science/article/...,Article,3,...,0,male,0.533653,"['female', 'male']","[0.6884891760488068, 0.8978163197000707]","Proton exchange membrane fuel cell, Reduced or...",United Kingdom,60015150,"[United Kingdom, United Kingdom]","[60015150, 60015150]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1922,Enhancing the performance of building integrat...,Solar Energy,85.0,8,10.1016/j.solener.2009.10.004,"['Philip C. Eames', 'Tapas Kumar Mallick', 'M....",2011,https://www.sciencedirect.com/science/article/...,Article,249,...,1821,male,0.941474,"['female', 'male', 'female', 'female', 'female...","[0.5279053453200885, 0.667434403771005, 0.6575...",Photovoltaics; Buildings; Solar concentration;...,Ireland,60012873,"[United Kingdom, United Kingdom, United Kingdo...","[112538988, 111017204, 104054219, 103600123, 6..."
1923,Theoretical investigation considering manufact...,Solar Energy,131.0,NaN,10.1016/j.solener.2016.02.050,"['Nabin Sarmah', 'J. P. Ferrer-Rodríguez', 'Se...",2016,https://www.sciencedirect.com/science/article/...,Article,37,...,133,male,0.529378,"['male', 'female', 'female', 'female', 'female...","[0.5271602862804456, 0.6711257863210672, 0.688...",Concentrator photovoltaics; Optics; Ray trace ...,United Kingdom,60026479,"[India, Spain, United Kingdom, India, Spain, U...","[103421874, 128766104, 60018186, 60025757, 600..."
1924,Increased photovoltaic performance through t

In [ ]:
data.to_csv("./everything_uk_genders_aff.csv", index = False)